In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

from models.fc_imv import FC_Marginal, JointVAE

## Example of a forward pass on fake data

In [3]:
# two arbitrary views, one with 1000 features, and another with 500 features
marg_model1 = FC_Marginal(input_size=1000, hidden_sizes=[128, 128], z_dim=10, prediction_dim=10)
marg_model2 = FC_Marginal(input_size=500, hidden_sizes=[64, 64], z_dim=10, prediction_dim=10)

# joint model
joint_model = JointVAE(marginal_models=[marg_model1, marg_model2], hidden_dim=128)

# define optimizer
optimizer = AdamW(joint_model.parameters(), lr=1e-3)

In [4]:
# fake inputs/targets
example_input1 = torch.randn(100, 1000)
example_input2 = torch.randn(100, 500)
example_target = torch.randint(0, 10, (100,))

In [5]:
# get the predictions and distribution of the joint model, and the predictions and distributions of the marginal models
yhat, poe_dist, yhats, dists = joint_model([example_input1, example_input2])

In [6]:
# pass the predictions and distributions to the loss function and update parameters
loss = joint_model.loss(example_target, yhat, poe_dist, yhats, dists)
loss.backward()

## Example with real data (lets try!)

In [7]:
import pandas as pd
import numpy as np

In [126]:
lipData = pd.read_csv('~/Documents/Data/example_omicsdata/multi-omics-from-share/ICL104_Pep_Prot_Lipid_Metab/ICL104_lipidpos_edata_log2_normalized.csv')
metabData = pd.read_csv('~/Documents/Data/example_omicsdata/multi-omics-from-share/ICL104_Pep_Prot_Lipid_Metab/ICL104_metab_edata_log2_normalized.csv')
proData = pd.read_csv('~/Documents/Data/example_omicsdata/multi-omics-from-share/ICL104_Pep_Prot_Lipid_Metab/ICL104_protein_edata_log2_normalized.csv')

fmeta = pd.read_csv('~/Documents/Data/example_omicsdata/multi-omics-from-share/ICL104_Pep_Prot_Lipid_Metab/ICL104_fmeta.csv')

In [127]:
# some samples missing from some datasets.
lipData.shape, metabData.shape, proData.shape

((250, 60), (133, 58), (3510, 61))

In [128]:
# filter all rows in fmeta where any of SampleID_proteomics, SampleID_proteomics, SampleID_metab are NA
fmeta = fmeta[~fmeta[['SampleID_proteomics', 'SampleID_lipidpos', 'SampleID_metab']].isna().any(axis=1)]

In [129]:
# make the "LIPID" column in lipData the index
lipData = lipData.set_index('LIPID')
metabData = metabData.set_index('METABOLITE')
proData = proData.set_index('REFERENCE')

In [135]:
# get the elements in lipData.columns that are in fmeta['SampleID_lipidpos']
lipData = lipData.loc[:,lipData.columns.isin(fmeta['SampleID_lipidpos'])]
metabData = metabData.loc[:,metabData.columns.isin(fmeta['SampleID_metab'])]
proData = proData.loc[:,proData.columns.isin(fmeta['SampleID_proteomics'])]

# impute the datasets with the row means
lipData = lipData.apply(lambda row: row.fillna(row.mean()), axis=1)
proData = proData.apply(lambda row: row.fillna(row.mean()), axis=1)
metabData = metabData.apply(lambda row: row.fillna(row.mean()), axis=1)

In [139]:
# drop rows with all NA values
lipData = lipData.dropna(how='all')
proData = proData.dropna(how='all')
metabData = metabData.dropna(how='all')

In [140]:
assert all(lipData.columns == fmeta['SampleID_lipidpos'])
assert all(metabData.columns == fmeta['SampleID_metab'])
assert all(proData.columns == fmeta['SampleID_proteomics'])

In [141]:
y = fmeta.Virus

In [154]:
fmeta['Virus_y'] = fmeta.Virus.replace({'Cal[0-9]*': 'Cal'}, regex=True)
y = fmeta.Virus_y

# Construct the Model

In [145]:
# three view-specific encoders with the appropriate input size:
lip_marg = FC_Marginal(input_size=lipData.shape[0], hidden_sizes=[128, 128], z_dim=10, prediction_dim=y.nunique())
metab_marg = FC_Marginal(input_size=metabData.shape[0], hidden_sizes=[64, 64], z_dim=10, prediction_dim=y.nunique())
pro_marg = FC_Marginal(input_size=proData.shape[0], hidden_sizes=[256, 128], z_dim=10, prediction_dim=y.nunique())

# joint model
joint_model = JointVAE(marginal_models=[lip_marg, metab_marg, pro_marg], hidden_dim=128)

# define optimizer
optimizer = AdamW(joint_model.parameters(), lr=1e-3)

In [161]:
# turn data into tensors
lip_tensor = torch.tensor(lipData.T.values, dtype=torch.float32)
pro_tensor = torch.tensor(proData.T.values, dtype=torch.float32)
metab_tensor = torch.tensor(metabData.T.values, dtype=torch.float32)

y_gt = y.astype('category').cat.codes
y_gt = torch.tensor(y_gt.values, dtype=torch.int64)

for i in range(1000):
    # get the predictions and distribution of the joint model, and the predictions and distributions of the marginal models
    yhat, poe_dist, yhats, dists = joint_model([lip_tensor, metab_tensor, pro_tensor])

    # pass the predictions and distributions to the loss function and update parameters
    loss = joint_model.loss(y_gt, yhat, poe_dist, yhats, dists)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'Epoch {i+1} loss: {loss.item():.3f}')

Epoch 1 loss: 32312.787
Epoch 2 loss: 7094.718
Epoch 3 loss: 2351.911
Epoch 4 loss: 1066.952
Epoch 5 loss: 666.980
Epoch 6 loss: 483.943
Epoch 7 loss: 390.422
Epoch 8 loss: 349.716
Epoch 9 loss: 331.724
Epoch 10 loss: 345.406
Epoch 11 loss: 368.139
Epoch 12 loss: 398.681
Epoch 13 loss: 424.238
Epoch 14 loss: 444.353
Epoch 15 loss: 449.555
Epoch 16 loss: 437.142
Epoch 17 loss: 409.959
Epoch 18 loss: 377.732
Epoch 19 loss: 348.670
Epoch 20 loss: 325.484
Epoch 21 loss: 318.492
Epoch 22 loss: 300.285
Epoch 23 loss: 297.789
Epoch 24 loss: 284.085
Epoch 25 loss: 278.555
Epoch 26 loss: 269.836
Epoch 27 loss: 266.823
Epoch 28 loss: 258.777
Epoch 29 loss: 254.606
Epoch 30 loss: 252.476
Epoch 31 loss: 248.936
Epoch 32 loss: 241.565
Epoch 33 loss: 240.438
Epoch 34 loss: 235.067
Epoch 35 loss: 233.955
Epoch 36 loss: 228.879
Epoch 37 loss: 215.402
Epoch 38 loss: 213.239
Epoch 39 loss: 199.681
Epoch 40 loss: 195.106
Epoch 41 loss: 195.111
Epoch 42 loss: 190.075
Epoch 43 loss: 187.693
Epoch 44 loss: 

In [165]:
yhat.argmax(dim=1) == y_gt

tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True, False,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True])